> 在线讲义：[第 2 章 · 把动作跟踪写成 MDP：看什么、做什么、怎么打分、什么时候算摔](https://wikieden.github.io/robot-motion-rl/lessons/02-tracking-mdp.html) · 本书《机器人运控与强化学习》

# 第 2 章 · 把动作跟踪写成 MDP：看什么、做什么、怎么打分、什么时候算摔

配套讲义：仓库里 `robot-motion-rl/notes/lessons/02-tracking-mdp.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（误差是编的，σ 和权重取自 SuperMimic 表 A.2）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 观测 | 93 × 10 | `obs_dim()` | 930 |
| 第三步 奖励核 | $\exp(-e^2/\sigma^2)$ | `kernel(e, 0.3)` | 1.000、0.895、0.368、0.018 |
| 第四步 倾斜 | $|1-\cos\theta|$ | `tilt_error_from_angle(78)` | 0.792 |
| 第三步 总奖励 | 加权 + 正则 × dt | `tracking_reward(...)` | 任务 4.193，正则 −0.005，r = 0.0838 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
OBS_TERMS = {"关节位置 q−q0": 29, "关节速度 q̇": 29, "IMU 角速度": 3, "重力投影": 3, "上一步动作": 29}

**函数 `obs_dim`**：本体感知观测维度：每帧 5 项相加，再乘历史帧数。

In [ ]:
def obs_dim(history=10):
    """本体感知观测维度：每帧 5 项相加，再乘历史帧数。"""
    return sum(OBS_TERMS.values()) * history

**函数 `kernel`**：跟踪奖励核：exp(−e²/σ²)，误差 0 时得 1，误差等于 σ 时得 e^−1 ≈ 0.368。

In [ ]:
def kernel(err, sigma):
    """跟踪奖励核：exp(−e²/σ²)，误差 0 时得 1，误差等于 σ 时得 e^−1 ≈ 0.368。"""
    return float(np.exp(-(err**2) / sigma**2))

**函数 `tilt_error_from_angle`**：直立参考下，机器人倾斜 deg 度时的倾斜误差：|1 − cos(角度)|。

In [ ]:
def tilt_error_from_angle(deg):
    """直立参考下，机器人倾斜 deg 度时的倾斜误差：|1 − cos(角度)|。"""
    return abs(1 - np.cos(np.radians(deg)))

**函数 `tracking_reward`**：BeyondMimic 式每步奖励（SuperMimic 表 A.2 的权重），每项再乘控制周期 dt。

In [ ]:
def tracking_reward(pos_err, ori_err, lin_err, ang_err, root_pos_err, root_ori_err, da, joint_violation, dt=0.02):
    """BeyondMimic 式每步奖励（SuperMimic 表 A.2 的权重），每项再乘控制周期 dt。"""
    task = (
        kernel(pos_err, 0.3)
        + kernel(ori_err, 0.4)
        + kernel(lin_err, 1.0)
        + kernel(ang_err, 3.14)
        + 0.5 * kernel(root_pos_err, 0.3)
        + 0.5 * kernel(root_ori_err, 0.4)
    )
    reg = -0.1 * float(np.sum(np.square(da))) - 2.0 * joint_violation
    return (task + reg) * dt, task, reg

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 观测维度")
print("每帧", sum(OBS_TERMS.values()), "维 × 10 帧 =", obs_dim(), "维")

In [ ]:
print("\n第二步 奖励核")
for e in (0.0, 0.1, 0.3, 0.6):
    print(f"位置误差 {e} m，σ=0.3 → {kernel(e, 0.3):.3f}")

In [ ]:
print("\n第三步 终止")
print(f"倾斜 78° → 倾斜误差 {tilt_error_from_angle(78):.3f}（阈值 0.8）")

In [ ]:
print("\n第四步 一步的总奖励")
r, task, reg = tracking_reward(0.1, 0.2, 0.5, 1.0, 0.1, 0.2, np.array([0.2, -0.1]), 0.0)
print(f"任务项 {task:.3f}，正则项 {reg:.3f}，乘 dt=0.02 后 r = {r:.4f}")

## 3. 代码里才看得到的

- 论文里每个刚体项是「先对所有被跟踪刚体的误差平方求平均，再进核」；代码为了好算，直接给一个平均后的误差。
- 角速度项的 σ 是 3.14（约 π rad/s），所以角速度误差 1.0 几乎不扣分（0.904），这一项其实很宽松。

## 4. 自己改一改

- 把位置项 σ 从 0.3 改成 0.1，看同样 0.1 m 的误差得分掉到多少。
- 把 dt 去掉，看每步奖励的量级变化（为什么各项要乘 dt：让奖励和控制频率无关）。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 闭眼跳舞靠本体：关节、速度、IMU、上一步，十帧历史补看不见；
> 跟着老师看十帧，输出二十九个设定值；
> 误差进核零到一，加权求和乘 dt；
> 偏高倒斜就终止，正分拿不到，自然学会别摔。